# 4단계 · DistilBERT 파인튜닝 (프로젝트의 핵심)

**무엇을 하나요?** 이미 영어를 잘 이해하는 작은 트랜스포머 모델 **DistilBERT**를 가져와, 우리 657개 학습 문장으로 *of/from/with 분류*에 맞게 **재교육(파인튜닝)**합니다. 도구는 **PyTorch + HuggingFace Transformers**.

이번 단계에서 새로 만나는 개념:
- **토크나이저(tokenizer)** — 문장을 모델이 읽을 수 있는 숫자 조각으로 쪼개는 도구
- **사전학습 모델 + 분류 헤드** — 언어를 아는 본체 위에 'of/from/with 셋 중 하나 고르기' 판단층을 새로 붙임
- **Trainer** — 학습 과정을 자동으로 굴려주는 HuggingFace의 학습 관리자
- **에폭(epoch)** — 학습 데이터를 처음부터 끝까지 도는 횟수 (여기선 5번)
- **클래스 가중치(class weight)** — 소수 클래스(from/with)를 더 신경 쓰게 만드는 불균형 보정

**목표:** 3단계 베이스라인(Macro-F1 0.525)을 넘기기.

In [1]:
import numpy as np                 # 숫자 배열 계산
import pandas as pd                # 표(CSV) 데이터 처리
import torch                       # PyTorch: 학습 엔진
from datasets import Dataset       # 문장 데이터를 학습에 편한 형태로 담는 그릇
from transformers import (
    AutoTokenizer,                          # 문장을 숫자 조각으로 쪼개는 토크나이저 로더
    AutoModelForSequenceClassification,     # '문장 분류용' 모델 로더
    TrainingArguments,                      # 학습 설정(에폭/배치크기/학습률 등)을 담는 통
    Trainer,                                # 학습을 자동으로 굴려주는 관리자
    DataCollatorWithPadding,                # 배치 안 문장 길이를 자동으로 맞춰(패딩)주는 도구
    set_seed,                               # 무작위성을 고정해 결과를 재현 가능하게
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight  # 클래스 가중치 자동 계산

set_seed(42)  # 항상 같은 결과가 나오도록 시드 고정(연구 재현성)
print("torch:", torch.__version__, "| MPS(M4 가속) 사용 가능:", torch.backends.mps.is_available())

/Users/jamie/anaconda3/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.12.0 | MPS(M4 가속) 사용 가능: True


In [2]:
# of/from/with를 모델이 다루는 숫자(0/1/2)로 바꾸는 사전을 만듭니다. 모델은 글자가 아니라 숫자로 학습하기 때문.
label2id = {"of": 0, "from": 1, "with": 2}
id2label = {v: k for k, v in label2id.items()}  # 반대 방향(숫자->글자) 사전. 나중에 결과 해석에 사용

# 2단계에서 만든 학습/검증 분할을 불러옵니다.
train_df = pd.read_csv("../data/splits/train.csv")
val_df   = pd.read_csv("../data/splits/val.csv")

# 글자 라벨(of/from/with)을 숫자(0/1/2) 열 'labels'로 추가합니다. Trainer는 'labels'라는 이름을 기대합니다.
train_df["labels"] = train_df["label"].map(label2id)
val_df["labels"]   = val_df["label"].map(label2id)

print("학습:", len(train_df), "검증:", len(val_df))

학습: 657 검증: 141


### 토크나이저 — 문장을 숫자로 쪼개기

모델은 글자를 직접 못 읽습니다. 토크나이저가 `"made ___ wood"` 같은 문장을 `[101, 2081, ..., 102]` 같은 **숫자 조각(토큰 ID)**으로 바꿔줍니다. DistilBERT가 학습될 때 쓰던 것과 **똑같은** 토크나이저를 써야 모델이 알아듣습니다.

In [3]:
MODEL = "distilbert-base-uncased"   # 사용할 사전학습 모델 이름(BERT의 작고 빠른 버전, 소문자 처리판)
tokenizer = AutoTokenizer.from_pretrained(MODEL)  # 그 모델에 맞는 토크나이저를 내려받아 준비

# 한 문장을 토크나이저에 넣으면 어떻게 변하는지 예시로 확인합니다.
ex = train_df["sentence_masked"].iloc[2]
print("원문 :", ex)
print("토큰 :", tokenizer.tokenize(ex)[:15], "...")  # 사람이 보기 위한 '조각 단어' 형태(앞 15개만)

원문 : For FRSCCM made ___ fiber factors higher than Fsub d , the thickness of suspending fluid covering fibers is not sufficient , thus leading to higher potential formation of fiber clumping and anisotropy in material performance .
토큰 : ['for', 'frs', '##cc', '##m', 'made', '_', '_', '_', 'fiber', 'factors', 'higher', 'than', 'f', '##su', '##b'] ...


In [4]:
# pandas 표를 HuggingFace Dataset(학습용 그릇)으로 바꿉니다. 필요한 두 열만 사용.
def make_dataset(df):
    ds = Dataset.from_pandas(df[["sentence_masked", "labels"]], preserve_index=False)
    # 모든 문장을 토큰 숫자로 변환. truncation=True, max_length=128: 너무 긴 문장은 128토큰에서 자름
    return ds.map(lambda b: tokenizer(b["sentence_masked"], truncation=True, max_length=128), batched=True)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)
train_ds  # input_ids(숫자조각), attention_mask(실제토큰 표시) 열이 추가된 걸 확인

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

Map: 100%|██████████| 657/657 [00:00<00:00, 13718.07 examples/s]

Map:   0%|          | 0/141 [00:00<?, ? examples/s]

Map: 100%|██████████| 141/141 [00:00<00:00, 15930.74 examples/s]

Dataset({
    features: ['sentence_masked', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 657
})

### 모델 불러오기

DistilBERT 본체(언어 이해) 위에 **새 분류 헤드(of/from/with 셋 중 하나 선택)**를 붙입니다.

> 불러올 때 `MISSING`(분류 헤드 = 새로 만들어 우리가 학습할 부분)과 `UNEXPECTED`(원래의 단어맞히기 헤드 = 우리에겐 불필요해 버림) 보고가 뜨는데, **이건 정상**입니다. "언어는 알지만 이 분류 업무는 아직 모르는" 상태라는 뜻이에요.

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,          # 분류할 종류 수: of/from/with → 3개
    id2label=id2label,     # 결과를 사람이 읽는 글자로 되돌리기 위한 사전
    label2id=label2id,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 11060.93it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 불균형 보정 — 클래스 가중치

of가 54%로 많고 with가 19%로 적습니다. 그냥 두면 모델이 "of로만 찍는" 편한 길로 샐 수 있어요. 그래서 **적은 클래스의 실수에 더 큰 벌점**을 주도록 가중치를 계산해 손실(loss) 계산에 반영합니다.

In [6]:
# 'balanced': 적은 클래스일수록 큰 가중치를 자동 계산 (of<from<with 순으로 커짐)
class_weights = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_df["labels"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float)  # PyTorch가 쓰는 텐서 형태로 변환
print("클래스 가중치 [of, from, with]:", [round(w, 3) for w in class_weights.tolist()])

# 기본 Trainer는 가중치 없는 손실을 씁니다. 가중치를 반영하려고 손실 계산만 살짝 바꾼 Trainer를 만듭니다.
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights              # 가중치를 보관
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")                   # 정답을 따로 빼두고
        outputs = model(**inputs)                       # 모델이 예측(logits) 생성
        # 가중치를 적용한 교차엔트로피 손실: 틀린 정도를 클래스별 가중치로 보정해 계산
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

클래스 가중치 [of, from, with]: [0.613, 1.259, 1.738]


In [7]:
# 학습 도중/이후 성적을 어떻게 잴지 정의합니다: 정확도와 Macro-F1.
def compute_metrics(eval_pred):
    logits, labels = eval_pred              # 모델의 원점수(logits)와 정답
    preds = np.argmax(logits, axis=-1)      # 가장 점수 높은 클래스를 최종 예측으로 선택
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),  # 불균형 핵심 지표
    }

In [8]:
# 학습 설정을 모읍니다. 각 항목이 '어떻게 학습할지'를 정합니다.
args = TrainingArguments(
    output_dir="../outputs/distilbert",      # 체크포인트/로그 저장 위치(깃에는 제외됨)
    num_train_epochs=5,                       # 학습 데이터를 5번 반복 학습
    per_device_train_batch_size=16,           # 한 번에 16문장씩 묶어서 학습
    per_device_eval_batch_size=32,            # 평가는 32문장씩
    learning_rate=2e-5,                        # 학습 보폭(트랜스포머 파인튜닝의 일반적 값)
    eval_strategy="epoch",                    # 매 에폭 끝에 검증셋으로 평가
    save_strategy="epoch",                    # 매 에폭 끝에 저장
    load_best_model_at_end=True,              # 끝나면 '가장 좋았던 에폭'의 모델을 최종 채택
    metric_for_best_model="f1_macro",         # '가장 좋음'의 기준 = Macro-F1
    greater_is_better=True,                   # 그 지표는 클수록 좋음
    save_total_limit=1,                        # 디스크 절약: 체크포인트는 최신 1개만 유지
    logging_steps=20,                          # 20스텝마다 학습 손실 기록
    disable_tqdm=True,                         # 진행바 끄기(노트북 출력을 깔끔하게)
    seed=42,
    report_to="none",                         # 외부 로깅 도구(wandb 등) 사용 안 함
)

# 위에서 만든 가중치 Trainer에 모델/데이터/설정/평가함수를 모두 연결합니다.
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,                          # 토크나이저 연결
    data_collator=DataCollatorWithPadding(tokenizer),    # 배치 길이 자동 맞춤
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [9]:
# 실제 학습을 시작합니다. M4 칩에서 5에폭에 약 2분 안팎 걸립니다.
# 학습이 잘 되면 손실(loss)이 점점 내려가고, 검증 F1이 올라갑니다.
trainer.train()

/Users/jamie/anaconda3/envs/default/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'loss': '1.091', 'grad_norm': '1.636', 'learning_rate': '1.819e-05', 'epoch': '0.4762'}


{'loss': '1.053', 'grad_norm': '2.71', 'learning_rate': '1.629e-05', 'epoch': '0.9524'}


{'eval_loss': '1.019', 'eval_accuracy': '0.5745', 'eval_f1_macro': '0.5617', 'eval_runtime': '0.8165', 'eval_samples_per_second': '172.7', 'eval_steps_per_second': '6.123', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.42it/s]

... (27 outputs left out)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.57it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.55it/s]

{'train_runtime': '81.04', 'train_samples_per_second': '40.54', 'train_steps_per_second': '2.591', 'train_loss': '0.8411', 'epoch': '5'}


TrainOutput(global_step=210, training_loss=0.8410990329015823, metrics={'train_runtime': 81.0397, 'train_samples_per_second': 40.536, 'train_steps_per_second': 2.591, 'train_loss': 0.8410990329015823, 'epoch': 5.0})

In [10]:
# 에폭별 검증 성적 변화를 표로 정리합니다(과적합 여부와 개선 추이를 한눈에).
history = [h for h in trainer.state.log_history if "eval_f1_macro" in h]
pd.DataFrame(history)[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]].round(4)

,epoch,eval_loss,eval_accuracy,eval_f1_macro
0,1.0,1.0187,0.5745,0.5617
1,2.0,0.8994,0.6099,0.5483
2,3.0,0.8554,0.6312,0.5875
3,4.0,0.8500,0.6383,0.5888
4,5.0,0.8525,0.6454,0.6017


In [11]:
# 최종 채택된(가장 좋았던) 모델로 검증셋을 다시 예측해 상세 성적을 봅니다.
pred_logits = trainer.predict(val_ds)                 # 검증셋 예측 결과(원점수 포함)
y_pred = np.argmax(pred_logits.predictions, axis=-1)  # 최종 예측 라벨(0/1/2)
y_true = val_df["labels"].values                      # 실제 정답

# 클래스별 정밀도/재현율/F1 상세표 (글자 라벨로 보기)
print(classification_report(y_true, y_pred, target_names=["of", "from", "with"], digits=3))

/Users/jamie/anaconda3/envs/default/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

          of      0.760     0.740     0.750        77
        from      0.500     0.459     0.479        37
        with      0.531     0.630     0.576        27

    accuracy                          0.645       141
   macro avg      0.597     0.610     0.602       141
weighted avg      0.648     0.645     0.646       141



In [12]:
# 혼동행렬: 무엇을 무엇으로 헷갈렸는지 (행=실제, 열=예측)
labels = ["of", "from", "with"]
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
pd.DataFrame(cm, index=[f"실제 {l}" for l in labels], columns=[f"예측 {l}" for l in labels])

,예측 of,예측 from,예측 with
실제 of,57,13,7
실제 from,12,17,8
실제 with,6,4,17


In [13]:
# 학습된 최종 모델과 토크나이저를 한 폴더에 저장합니다. 5단계 평가에서 이걸 불러 테스트셋에 적용합니다.
save_dir = "../outputs/distilbert-best"
trainer.save_model(save_dir)        # 모델 가중치 + 설정 저장
tokenizer.save_pretrained(save_dir) # 토크나이저도 함께 저장(짝이 맞아야 하므로)
print("저장 완료 ->", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.48it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.46it/s]

저장 완료 -> ../outputs/distilbert-best


## 정리

위 표의 **검증 Macro-F1**을 3단계 베이스라인과 비교하세요:

| 모델 | Macro-F1 (검증) |
|------|:---:|
| 다수 클래스 | ~0.235 |
| TF-IDF + LogReg | ~0.525 |
| **DistilBERT (이번)** | (위 결과) |

베이스라인을 넘었다면, **PyTorch + Transformers로 트랜스포머를 파인튜닝해 실제 성능을 끌어올린** 경험이 완성된 것입니다. 다음 5단계에서는 지금까지 한 번도 안 본 **테스트셋**으로 최종 점수를 확정하고 논문(39%)과 비교합니다.